# 03 · Clients, servers, and data across boundaries

Two programs and a port; HTTP as text on a socket; your own API and your own client library;
someone else's API; failure, on purpose; a database that's a file vs one that's a server; and the
formats data travels in. This is where your network background and the code meet.

**Services run in separate terminals.** A server is a process that blocks forever, waiting for
connections. That's a script's job, not a notebook cell's (callback to 02 §E: a script is its own
process). Each section that needs a service gives the command and checks whether it's up.

In [ ]:
%load_ext autoreload
%autoreload 2

import json, os, socket, subprocess, time
from pathlib import Path

import pandas as pd
import requests

from netlab import paths
from netlab.check import check

def is_up(host, port, timeout=0.5):
    """True if something accepts TCP connections on host:port."""
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

def load_env(path=paths.ROOT / ".env"):
    """Read KEY=value lines from .env into os.environ (values already set win). No library needed."""
    if not path.exists():
        return False
    for line in path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())
    return True

print(".env loaded" if load_env() else "no .env yet. In a terminal at the repo root:  cp .env.example .env")
INVENTORY_URL = os.environ.get("INVENTORY_URL", "http://127.0.0.1:8000")

## A. A connection is two programs and a port

`services/echo_server.py` is about 40 lines of standard-library `socket` code: it binds a port,
accepts a connection, and sends back whatever arrives. Open it and read the comments.

**Terminal.** In a new terminal at the repo root (leave it running):

```bash
uv run python services/echo_server.py
```

In [ ]:
if is_up(*("127.0.0.1", 9000)):
    print("echo server is up")
else:
    print("echo server is NOT running. Start it in a terminal at the repo root:  uv run python services/echo_server.py")

In [ ]:
try:
    with socket.create_connection(("127.0.0.1", 9000), timeout=2) as s:   # TCP handshake happens here
        s.sendall(b"hello")
        reply = s.recv(4096)
    print(reply, list(reply))
except OSError as e:
    print("couldn't connect:", type(e).__name__, "-", e)

Those are the bytes from 01 §B, on a real connection. The server's terminal logged your client's
address and source port: an ephemeral port your OS picked, exactly as in a 5-tuple.

**Terminal.** Which process is listening? In a terminal (or the cell below):

```bash
lsof -nP -iTCP:9000 -sTCP:LISTEN
```

In [ ]:
!lsof -nP -iTCP:9000 -sTCP:LISTEN

### Bind address

Stop the echo server (Ctrl+C) and restart it on every interface:
`uv run python services/echo_server.py --host 0.0.0.0`. macOS may ask whether to allow incoming
connections; that's the host firewall noticing a new listener. Then run the next cell.

In [ ]:
try:
    probe = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    probe.connect(("192.0.2.1", 9))      # UDP "connect" sends nothing; it only picks the outgoing interface
    LAN_IP = probe.getsockname()[0]
    probe.close()
except OSError:
    LAN_IP = None
print("this machine's LAN address:", LAN_IP)
for host in ["127.0.0.1"] + ([LAN_IP] if LAN_IP else []):
    print(f"  {host}:9000 ->", "accepting connections" if is_up(host, 9000) else "refused")

Bound to `127.0.0.1`, only this machine can connect: the LAN address is refused. Bound to `0.0.0.0`,
every interface accepts, including the one your LAN (and anything on it) can reach. That's the
difference between a dev tool and an exposed service, and why dev servers default to `127.0.0.1`.
Try it from your phone or another machine with `nc <LAN IP> 9000`. Then restart it on `127.0.0.1`.

### Exercise 1 — change the server

Edit `services/echo_server.py` (the `TODO (learner)` line) so it replies with the message
**uppercased, a space, then the server's current time** in ISO format, e.g.
`HELLO 2026-03-01T10:00:02.123456`. Restart the server, then run the check.

In [ ]:
from datetime import datetime

def echo_reply(message: str) -> str:
    with socket.create_connection(("127.0.0.1", 9000), timeout=2) as s:
        s.sendall(message.encode())
        return s.recv(4096).decode()

def server_reply_ok(message: str) -> bool:
    reply = echo_reply(message)
    if reply == message:
        raise NotImplementedError("the server still echoes unchanged")
    text, _, stamp = reply.rpartition(" ")
    datetime.fromisoformat(stamp)            # raises if the last word isn't an ISO timestamp
    return text == message.upper()

check(server_reply_ok, [(("hello",), True), (("show ip route",), True)]);

## B. HTTP is text on a socket

Open a TCP connection to `example.com` port 80 and type an HTTP request by hand.

In [ ]:
request = b"GET / HTTP/1.1\r\nHost: example.com\r\nConnection: close\r\n\r\n"
try:
    with socket.create_connection(("example.com", 80), timeout=5) as s:
        s.sendall(request)
        chunks = []
        while chunk := s.recv(4096):      # read until the server closes the connection
            chunks.append(chunk)
    response = b"".join(chunks)
    head, _, body = response.partition(b"\r\n\r\n")   # a blank line separates headers from body
    print(head.decode())
    print(f"\n... then {len(body)} bytes of body: {body[:60]!r}")
except OSError as e:
    print("no connection (offline?):", type(e).__name__, "-", e)

That's all HTTP is: a formatted string on a TCP connection. A request line, headers, a blank line,
an optional body. The response has the same shape, starting with a status line.

**Terminal.** The same exchange, annotated by curl (`>` is what you sent, `<` what came back):

```bash
curl -v http://example.com
```

In [ ]:
!curl -sv http://example.com -o /dev/null 2>&1 | head -25

In [ ]:
import urllib.request     # the standard library's HTTP client

with urllib.request.urlopen("http://example.com", timeout=5) as r:
    print(r.status, r.headers["Content-Type"], len(r.read()), "bytes")

In [ ]:
r = requests.get("http://example.com", timeout=5)
print(r.status_code)
print(dict(r.headers))
print("we sent:", dict(r.request.headers))
print(r.text[:120])

In [ ]:
# This cell fails on purpose: the body is HTML, not JSON
try:
    r.json()
except Exception as e:
    print(type(e).__name__, "-", e)

`requests` is convenience over the same mechanism: it builds the request text, manages the socket,
parses the status line and headers, and decodes the body. `.text` is the body as text; `.json()`
parses it as JSON, and fails if it isn't.

In [ ]:
r = requests.get("https://example.com", timeout=5)
print(r.status_code, r.url)

**HTTPS** is the same HTTP, inside TLS. After the TCP handshake, client and server do a TLS
handshake: the server presents a certificate, the client checks it chains to a trusted CA and
matches the hostname, and they agree on keys. Then the identical request text flows, encrypted.
`requests` verifies certificates by default. You'll see `verify=False` in examples online: it turns
that check off, which makes you open to interception. Never use it outside a throwaway lab.

## C. Your own API, and your own client library

`services/inventory_api.py` serves the notebook 01 topology as JSON, using only the standard library.
The devices are the same `Device` dataclass, turned into dicts with `asdict` (callback to 01 §E).

**Terminal.** In another new terminal at the repo root (leave it running):

```bash
uv run python services/inventory_api.py
```

In [ ]:
if is_up(*("127.0.0.1", 8000)):
    print("inventory API is up")
else:
    print("inventory API is NOT running. Start it in a terminal at the repo root:  uv run python services/inventory_api.py")

In [ ]:
!curl -si {INVENTORY_URL}/health
print()
!curl -s {INVENTORY_URL}/devices | head -c 300

In [ ]:
r = requests.get(f"{INVENTORY_URL}/devices", timeout=2)
devices = r.json()                    # JSON array -> Python list of dicts
print(r.status_code, r.headers["Content-Type"], len(devices), "devices")
devices[1]

### Exercise 2 — finish the server, then write the client

**First**, open `services/inventory_api.py` and write the two `TODO (learner)` handlers:
`GET /devices/<name>` (404 for unknown names) and `POST /flows`. Their docstrings say exactly what
to return. Restart the server after each change.

**Then** write `InventoryClient` in the next cell: methods that hide HTTP behind function calls.

In [ ]:
%%writefile ../src/netlab/client.py
"""A client library for the inventory API (notebook 03 §C)."""

import requests


class InventoryClient:
    """Talk to the inventory API with method calls instead of raw HTTP."""

    def __init__(self, base_url: str, timeout: float = 2.0):
        """Store base_url (strip any trailing "/") and timeout, and create one requests.Session
        in self.session to reuse for every call."""
        raise NotImplementedError("your turn")

    def health(self) -> dict:
        """GET /health and return the parsed JSON. Raise on an HTTP error status (raise_for_status)."""
        raise NotImplementedError("your turn")

    def devices(self) -> list[dict]:
        """GET /devices and return the list of device dicts. Raise on an HTTP error status."""
        raise NotImplementedError("your turn")

    def device(self, name: str) -> dict | None:
        """GET /devices/<name> and return the device dict, or None if the server answers 404.
        Raise on any other HTTP error status."""
        raise NotImplementedError("your turn")

    def post_flow(self, flow: dict) -> dict:
        """POST flow as JSON to /flows and return the parsed JSON reply. Raise on an HTTP error status."""
        raise NotImplementedError("your turn")


In [ ]:
from netlab.client import InventoryClient

check(lambda: InventoryClient(INVENTORY_URL).health(), [((), {"status": "ok"})], name="health")
check(lambda: sorted(d["name"] for d in InventoryClient(INVENTORY_URL).devices()),
      [((), ['acc-sw-01', 'acc-sw-02', 'acc-sw-03', 'core-rtr-01', 'core-rtr-02', 'dist-sw-01', 'dist-sw-02', 'edge-rtr', 'fw-01'])], name="devices")
check(lambda name: (InventoryClient(INVENTORY_URL).device(name) or {}).get("role"),
      [(("fw-01",), "firewall"), (("acc-sw-03",), "access")], name="device")
check(lambda name: InventoryClient(INVENTORY_URL).device(name), [(("nope",), None)], name="device (unknown)")
check(lambda flow: InventoryClient(INVENTORY_URL).post_flow(flow),
      [(({"src_ip": "10.0.0.5", "dst_ip": "93.184.216.34", "dst_port": 443, "bytes": 1532},), {"status": "stored"})],
      name="post_flow");

In [ ]:
received = paths.DATA_PROCESSED / "received_flows.jsonl"
print(received.read_text().splitlines()[-3:] if received.exists() else "nothing received yet")

You just wrote an **SDK**: a library that hides HTTP behind function calls. Whoever uses
`InventoryClient` never sees a URL, a status code, or a socket.

## D. Someone else's API, three ways

GitHub's public REST API, no account needed. **1.** curl, in a terminal or here:

In [ ]:
!curl -s https://api.github.com/repos/python/cpython | head -40

In [ ]:
r = requests.get("https://api.github.com/repos/python/cpython", timeout=10)
if r.ok:
    print("stars:", r.json()["stargazers_count"])
else:
    print("GitHub said", r.status_code, r.json().get("message"))
print("calls left this hour:", r.headers.get("X-RateLimit-Remaining"), "of", r.headers.get("X-RateLimit-Limit"))

**2.** `requests`, above. Unauthenticated calls are limited per IP address (60 an hour, at the time
of writing): that header is the server telling you your budget.

**3.** An SDK. Install two libraries (`psycopg` is for §F):

**Terminal.** In a terminal at the repo root, then restart this kernel and re-run the setup cell at the top:

```bash
uv add "psycopg[binary]" PyGithub
```

In [ ]:
from github import Github

try:
    print("stars:", Github().get_repo("python/cpython").stargazers_count)
except Exception as e:
    print(type(e).__name__, "-", e)

**Look underneath.** Where does PyGithub make its HTTP calls?

In [ ]:
import github
gh_pkg = Path(github.__file__).parent
print(gh_pkg)
print("files that import requests:", sorted(f.name for f in gh_pkg.glob("*.py") if "import requests" in f.read_text()))
for n, line in enumerate((gh_pkg / "Requester.py").read_text().splitlines(), 1):
    if line.startswith("import requests") or "requests.Session()" in line:
        print(f"Requester.py:{n}: {line.strip()}")

PyGithub is built on `requests`: its `Requester.py` creates a `requests.Session` and sends every
call through it. The SDK is someone's `client.py`, on your disk (callback to 01 §G). Yours has the
same shape.

**Tokens.** For authenticated calls (higher limits, private repos), create a GitHub token and put
it in `.env` as `GITHUB_TOKEN=...`. `.env` is gitignored; `.env.example` lists the name with a fake
value. Read it from the environment; never print it, and never paste it into a cell: the source *and*
the outputs of every cell are saved in the `.ipynb` (callback to 02 §E).

In [ ]:
token = os.environ.get("GITHUB_TOKEN", "")
has_real_token = bool(token) and not token.startswith("ghp_fake")
print("GITHUB_TOKEN is set:", has_real_token)          # say whether it's set, never what it is
if has_real_token:
    from github import Auth
    print("authenticated (remaining, limit):", Github(auth=Auth.Token(token)).rate_limiting)

## E. Failing on purpose

Across a boundary, failure is normal. The question is never *whether* a call fails, but *which*
failure it was and what you do about it. Each cell below causes one failure and prints its
exception type. (The inventory API must be running, except where a cell says otherwise.)

In [ ]:
def show(fn):
    """Run fn; print the exception's full type name if it raises."""
    try:
        result = fn()
        print("no exception:", result)
    except Exception as e:
        print(f"{type(e).__module__}.{type(e).__name__}: {str(e)[:120]}")

In [ ]:
# This cell fails on purpose: stop the inventory API (Ctrl+C), run this cell, then start it again
show(lambda: requests.get(f"{INVENTORY_URL}/health", timeout=2))

In [ ]:
# This cell fails on purpose: the wrong port (nothing listens on 8001)
show(lambda: requests.get("http://127.0.0.1:8001/health", timeout=2))

In [ ]:
# This cell fails on purpose: the server takes 2 s, we wait 0.5 s
show(lambda: requests.get(f"{INVENTORY_URL}/slow?seconds=2", timeout=0.5))

In [ ]:
# This cell fails on purpose: 404, turned into an exception by raise_for_status()
show(lambda: requests.get(f"{INVENTORY_URL}/nope", timeout=2).raise_for_status())

In [ ]:
# This cell fails on purpose: 500, a bug on the server's side
show(lambda: requests.get(f"{INVENTORY_URL}/boom", timeout=2).raise_for_status())

In [ ]:
# This cell fails on purpose: a 200 OK whose body isn't JSON
show(lambda: requests.get("http://example.com", timeout=5).json())

Fill this in from what you just saw (double-click to edit):

| Failure | Exception type | Layer (connection / HTTP / data) | Retry? |
|---|---|---|---|
| server stopped | | | |
| wrong port | | | |
| timeout | | | |
| 404 | | | |
| 500 | | | |
| not JSON | | | |

**Principles.** Catch the specific exception you can handle; never a bare `except:`, which also
swallows typos and Ctrl+C. Re-raise what you can't handle. Always set a timeout: the default is to
wait forever. Clean up with `with` / `finally`. Retry only what's safe to repeat: a `GET` is; a `POST`
may create a duplicate (APIs that care accept an *idempotency key* so a repeat is recognized).

### Exercise 3 — retry what's worth retrying

Connection errors, timeouts, 5xx and 429 (rate limited) are often temporary: retry them, waiting
longer each time so a struggling server gets room to recover. Other 4xx mean *your request* is
wrong; retrying just repeats the mistake.

In [ ]:
def fetch_with_retry(session, url, attempts=3, backoff=0.2):
    """GET url with session.get(url, timeout=2), retrying failures that are worth retrying.

    Retry: connection errors, timeouts, and HTTP 5xx or 429 responses (call raise_for_status()
    to turn an error status into an HTTPError). Don't retry other 4xx: raise at once.
    Wait backoff, then 2x backoff, then 4x backoff... between attempts (exponential backoff).
    Return the successful response. If every attempt fails, raise the last error.
    """
    raise NotImplementedError("your turn")

In [ ]:
class CountingSession(requests.Session):
    """A Session that counts the requests it sends, so the checks can see your retries."""
    def __init__(self):
        super().__init__()
        self.calls = 0

    def request(self, *args, **kwargs):
        self.calls += 1
        return super().request(*args, **kwargs)

def outcome(url, attempts=3):
    session = CountingSession()
    try:
        return (fetch_with_retry(session, url, attempts=attempts, backoff=0.01).status_code, "ok")
    except requests.RequestException as e:
        return (type(e).__name__, session.calls)

check(outcome, [
    ((f"{INVENTORY_URL}/flaky",), (200, "ok")),                    # fails twice, then succeeds
    ((f"{INVENTORY_URL}/boom",), ("HTTPError", 3)),                # 500 every time: 3 tries, then give up
    ((f"{INVENTORY_URL}/boom", 5), ("HTTPError", 5)),
    ((f"{INVENTORY_URL}/devices/nope",), ("HTTPError", 1)),        # 404: don't retry (needs your §C TODO)
    (("http://127.0.0.1:8001/health",), ("ConnectionError", 3)),   # nothing listening
]);

<details><summary>What people call this</summary>

Waiting longer between each retry is <b>exponential backoff</b>; adding randomness to it is <b>jitter</b>.<br>
An operation that's safe to repeat is <b>idempotent</b>.
</details>

## F. Two kinds of database

### SQLite: a database that's a file

`sqlite3` is in the standard library. Load the flows from notebook 02 and ask for the top talkers
in SQL. That's the third time you've solved this: dicts (01), pandas (01 and 02), now SQL.

In [ ]:
import sqlite3

flows_csv = paths.DATA_RAW / "flows.csv"
if not flows_csv.exists():
    print("data/raw/flows.csv is missing: run notebook 02 §C first")
flows = pd.read_csv(flows_csv)

paths.ensure_dirs()
db_path = paths.DATA_PROCESSED / "netlab.sqlite"
conn = sqlite3.connect(db_path)
flows.to_sql("flows", conn, if_exists="replace", index=False)

TOP_TALKERS_SQL = """
    SELECT src_ip, SUM(bytes) AS total_bytes
    FROM flows
    WHERE action = 'ALLOW'
    GROUP BY src_ip
    ORDER BY total_bytes DESC
    LIMIT 5
"""
pd.read_sql_query(TOP_TALKERS_SQL, conn)

In [ ]:
!lsof -nP {db_path}
print(f"database size: {db_path.stat().st_size:,} bytes; no port, no server")

The only process with the database open is this kernel. There's no server and no port: SQLite is a
library inside your process, reading and writing a file. Nothing to start, nothing to expose.

### Postgres in Docker: a database that's a server

Postgres is a separate program you connect to over TCP. Docker runs it in a container, so you don't
install it on your laptop. `services/docker-compose.yml` describes it; every line is commented.

In [ ]:
def docker_running() -> bool:
    try:
        return subprocess.run(["docker", "info"], capture_output=True, timeout=30).returncode == 0
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return False

DOCKER_OK = docker_running() and (paths.ROOT / ".env").exists()
if DOCKER_OK:
    print("Docker is running; .env exists")
else:
    print("Skipping the Postgres part of §F: Docker isn't running (or .env is missing).")
    print("  Install Docker Desktop: https://www.docker.com/products/docker-desktop/  then start it.")
    print("  Then: cp .env.example .env (if needed), restart this kernel, and re-run this section.")

**Terminal.** In a terminal at the repo root (the cell below runs the same command for you):

```bash
docker compose -f services/docker-compose.yml --env-file .env up -d
docker ps
```

In [ ]:
COMPOSE = ["docker", "compose", "-f", str(paths.ROOT / "services/docker-compose.yml"),
           "--env-file", str(paths.ROOT / ".env")]

def compose(*args):
    """Run a docker compose command for this project and show its output."""
    r = subprocess.run(COMPOSE + list(args), cwd=paths.ROOT, capture_output=True, text=True, timeout=300)
    print((r.stdout + r.stderr).strip()[-800:])
    return r.returncode

if DOCKER_OK:
    compose("up", "-d", "--wait")         # --wait: return once the healthcheck passes
    !docker ps --filter name=netlab
else:
    print("skipped: Docker not available")

In [ ]:
if DOCKER_OK:
    import psycopg

    e = os.environ
    PG_URL = e.get("DATABASE_URL") or (f"postgresql://{e['POSTGRES_USER']}:{e['POSTGRES_PASSWORD']}"
                                       f"@127.0.0.1:{e.get('POSTGRES_PORT', '5432')}/{e['POSTGRES_DB']}")
    password = e.get("POSTGRES_PASSWORD")
    print("connecting to", PG_URL.replace(password, "****") if password else PG_URL)   # never print a password

    typed = flows.astype({"bytes": "Int64"})                  # back to whole numbers (CSV made them floats)
    rows = [tuple(None if pd.isna(v) else v for v in row)     # missing -> None, which becomes SQL NULL
            for row in typed.itertuples(index=False)]
    with psycopg.connect(PG_URL, connect_timeout=5) as pg:
        pg.execute("DROP TABLE IF EXISTS flows")
        pg.execute("""CREATE TABLE flows (ts TEXT, src_ip TEXT, dst_ip TEXT, dst_port INT, proto TEXT,
                                           bytes BIGINT, action TEXT, device_id INT)""")
        with pg.cursor().copy("COPY flows FROM STDIN") as copy:     # bulk load: the fast path
            for row in rows:
                copy.write_row(row)
        print(pg.execute("SELECT COUNT(*) FROM flows").fetchone()[0], "rows loaded")
        print(pd.DataFrame(pg.execute(TOP_TALKERS_SQL).fetchall(), columns=["src_ip", "total_bytes"]))
else:
    print("skipped: Docker not available")

Same query, same answer, different machine: this time your process sent SQL over a TCP connection
to another program, which ran it and sent rows back.

**Container networking, in your terms:**
- `127.0.0.1:5432:5432` in the compose file is **NAT**: host port 5432 is forwarded to port 5432
  inside the container. Bound to `127.0.0.1`, so the LAN can't reach it (callback to §A).
- Inside a container, `localhost` means *the container itself*, not your laptop. A common bug: an
  app in one container connecting to `localhost:5432` finds nothing.
- Containers in the same compose project share a private network and reach each other **by service
  name**, through Docker's internal DNS: another container would connect to `db:5432`.

In [ ]:
# This cell fails on purpose: the database is stopped mid-session
if DOCKER_OK:
    compose("stop")
    try:
        psycopg.connect(PG_URL, connect_timeout=3)
    except psycopg.OperationalError as err:
        print("psycopg.OperationalError -", str(err).splitlines()[0])
    compose("up", "-d", "--wait")
else:
    print("skipped: Docker not available")

In [ ]:
if DOCKER_OK:
    compose("down")                 # containers removed; the named volume stays
    compose("up", "-d", "--wait")
    with psycopg.connect(PG_URL) as pg:
        print("after down + up:", pg.execute("SELECT COUNT(*) FROM flows").fetchone()[0], "rows: still there")
    compose("down", "-v")           # -v also deletes the volume
    compose("up", "-d", "--wait")
    with psycopg.connect(PG_URL) as pg:
        try:
            pg.execute("SELECT COUNT(*) FROM flows")
        except psycopg.errors.UndefinedTable as err:
            print("after down -v + up:", type(err).__name__, "- the data is gone")
else:
    print("skipped: Docker not available")

The container is disposable; the volume is where the data lives. When you're done with Postgres:
`docker compose -f services/docker-compose.yml --env-file .env down`.

| | SQLite | Postgres |
|---|---|---|
| Where the data lives | one file in `data/processed/` | a Docker volume, managed by the server |
| Who can connect | any process that can open the file | anything that can reach the port and log in |
| Writers at once | one at a time | many |
| What can fail | disk, file locks | everything in §E, plus the server and the container |
| What you operate | nothing | a server: start, stop, upgrade, back up, secure |

SQLite is the right answer more often than people assume: single-user tools, local caches,
embedded apps, and anything that doesn't need many writers over a network.

**Stretch:** add `inventory_api` to the compose file as a second service that talks to Postgres at
hostname `db`.

## G. Moving data between systems

The same 100,000 flows from 02 §C, generated fresh with their proper types, written three ways.

In [ ]:
from netlab.flows import generate_flows      # your module from 02 §E

df = generate_flows(100_000, seed=42)
fmt_dir = paths.DATA_PROCESSED / "formats"
fmt_dir.mkdir(parents=True, exist_ok=True)

FORMATS = {
    "csv":     (lambda p: df.to_csv(p, index=False),     pd.read_csv),
    "parquet": (lambda p: df.to_parquet(p, index=False), pd.read_parquet),
    "jsonl":   (lambda p: df.to_json(p, orient="records", lines=True, date_format="iso"),
                lambda p: pd.read_json(p, lines=True)),
}
results, read_back = {}, {}
for name, (write, read) in FORMATS.items():
    path = fmt_dir / f"flows.{name}"
    t0 = time.perf_counter(); write(path); t_write = time.perf_counter() - t0
    t0 = time.perf_counter(); read_back[name] = read(path); t_read = time.perf_counter() - t0
    results[name] = {"size (MB)": path.stat().st_size / 1e6, "write (s)": t_write, "read (s)": t_read}
pd.DataFrame(results).T.round(3)

In [ ]:
pd.DataFrame({"original": df.dtypes.astype(str), **{n: d.dtypes.astype(str) for n, d in read_back.items()}})

In [ ]:
for name, d in read_back.items():
    print(f"{name:>8}: device_id {d['device_id'].iloc[0]!r:>8}   ts {str(d['ts'].iloc[0])!r}")

CSV lost the timestamps' type, turned `bytes` (an int with gaps) into a float, and stripped
`device_id`'s leading zeros. JSON lines lost the same three: JSON has strings and numbers but no
dates, and the reader guessed the rest. Parquet kept all of it (a timestamp is still a timestamp),
because **the schema travels with the file**: column names *and* types are written into it.

In [ ]:
cols = ["src_ip", "bytes"]
t0 = time.perf_counter(); pd.read_parquet(fmt_dir / "flows.parquet", columns=cols); t_parquet = time.perf_counter() - t0
t0 = time.perf_counter(); pd.read_csv(fmt_dir / "flows.csv", usecols=cols); t_csv = time.perf_counter() - t0
print(f"two columns from Parquet: {t_parquet * 1e3:6.1f} ms")
print(f"two columns from CSV:     {t_csv * 1e3:6.1f} ms")

CSV is a row layout: to get two columns, the reader still has to scan every byte of every line.
Parquet is a column layout, so it reads just those two columns' blocks (callback to 01 §D).

**When to use each:** CSV for humans, spreadsheets and tools that only take CSV. Parquet for
pipelines and anything large or typed. JSON for API payloads and small nested records.

**Stretch:** add `GET /flows.parquet` to `inventory_api.py` and read it straight from the URL with
`pl.read_parquet(url)`.

## H. What the network does to your code

### Latency

In [ ]:
def time_calls(url, n=20):
    t0 = time.perf_counter()
    for _ in range(n):
        requests.get(url, timeout=10)
    return time.perf_counter() - t0

local = time_calls(f"{INVENTORY_URL}/health")
remote = time_calls("https://example.com")
print(f"20 calls to localhost:   {local:6.2f} s  ({local / 20 * 1e3:6.1f} ms each)")
print(f"20 calls to example.com: {remote:6.2f} s  ({remote / 20 * 1e3:6.1f} ms each)")

Same code, same work; the difference is distance. A loop of network calls is slow because of round
trips, not computation. Where an API allows it, ask for many things in one request.

### Waiting in parallel

In [ ]:
from concurrent.futures import ThreadPoolExecutor

slow_url = f"{INVENTORY_URL}/slow?seconds=0.5"
t0 = time.perf_counter()
for _ in range(20):
    requests.get(slow_url, timeout=5)
sequential = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=20) as pool:
    list(pool.map(lambda _: requests.get(slow_url, timeout=5), range(20)))
threaded = time.perf_counter() - t0
print(f"20 slow calls, one after another: {sequential:5.2f} s")
print(f"20 slow calls, 20 threads:        {threaded:5.2f} s")

This is 02 §F's I/O-bound row, with real HTTP: the threads spend their time waiting, so they can all
wait at once. (The server handles them in parallel too: it's a `ThreadingHTTPServer`.)

### Connection reuse

In [ ]:
def bare(url, n):
    t0 = time.perf_counter()
    for _ in range(n):
        requests.get(url, timeout=10)           # new TCP (and TLS) connection every call
    return time.perf_counter() - t0

def pooled(url, n):
    t0 = time.perf_counter()
    with requests.Session() as s:
        for _ in range(n):
            s.get(url, timeout=10)              # one connection, kept open and reused
    return time.perf_counter() - t0

for url, n in [(f"{INVENTORY_URL}/health", 50), ("https://example.com", 10)]:
    print(f"{n} calls to {url}: bare {bare(url, n):.2f} s, Session {pooled(url, n):.2f} s")

A `Session` keeps the TCP connection (and, for HTTPS, the TLS session) open between calls, so
later calls skip the handshakes. On localhost a handshake is nearly free; across the internet each
TLS handshake costs round trips. That's why your `InventoryClient` holds one Session.

### Name resolution

In [ ]:
for family, _, _, _, addr in socket.getaddrinfo("example.com", 443, proto=socket.IPPROTO_TCP):
    print(family.name, addr)

`getaddrinfo` is what every client calls before connecting. It consults `/etc/hosts` before DNS, so
a line like `127.0.0.1 inventory.local` lets you point a name at your own machine during development.

### Configuration

In [ ]:
url = os.environ.get("INVENTORY_URL", "http://127.0.0.1:8000")    # the environment wins; the default is for local dev
print(url)

Hosts, ports, credentials and URLs come from environment variables with local-dev defaults, never
hardcoded. The same code then runs on your laptop, in a container, and in production: only the
environment changes.

## I. Capstone

Write `scripts/collect.py`. Keep the logic in `src/netlab/` (say `src/netlab/collect.py` with a
`main()`), and keep the script thin, like `make_flows.py`. It must:

- [ ] read `INVENTORY_URL` from the environment, with a sensible default;
- [ ] use your `InventoryClient` and `fetch_with_retry` to pull all devices (move `fetch_with_retry`
      from §E into `src/netlab/client.py` first);
- [ ] write them to SQLite (`data/processed/netlab.sqlite`), or to Postgres if `DATABASE_URL` is set;
- [ ] export `data/processed/devices.parquet`;
- [ ] exit `0` on success, and nonzero with a one-line error message on failure;
- [ ] have a `--help`;
- [ ] be added to `README.md` under "How to run", with the services it needs.

In [ ]:
script = paths.ROOT / "scripts" / "collect.py"
if not script.exists():
    print("NOT YET: scripts/collect.py doesn't exist")
else:
    runs = [("--help", INVENTORY_URL, ["--help"]),
            ("API up", INVENTORY_URL, []),
            ("API down", "http://127.0.0.1:8001", [])]   # a URL where nothing listens = the API is down, from the script's view
    for label, url, extra in runs:
        env = {**os.environ, "INVENTORY_URL": url}
        env.pop("DATABASE_URL", None)                   # test the SQLite path
        r = subprocess.run(["uv", "run", "python", str(script), *extra], cwd=paths.ROOT, env=env,
                           capture_output=True, text=True, timeout=120)
        print(f"{label}: exit code {r.returncode}")
        for stream, text in [("stdout", r.stdout), ("stderr", r.stderr)]:
            if text.strip():
                print(f"  {stream}: " + text.strip().splitlines()[-1][:150])
    parquet = paths.DATA_PROCESSED / "devices.parquet"
    print("devices.parquet:", f"{len(pd.read_parquet(parquet))} rows" if parquet.exists() else "missing")

Expect: `--help` exits 0, "API up" exits 0, "API down" exits nonzero with a one-line message.

## What you can do now

- Open a TCP connection, send bytes, and choose a bind address on purpose.
- Speak HTTP by hand, with curl, with the standard library, and with `requests`.
- Serve an API, write a client library for it, and read someone else's SDK to see how it works.
- Name the failure (connection, HTTP, data) and decide whether retrying it is safe.
- Use a database that's a file and one that's a server in a container, and say when each fits.
- Pick a data format by what it keeps and what it costs, and measure network effects in your code.

## Glossary: what people call these

| Term | Meaning, in this notebook's terms |
|---|---|
| socket | one endpoint of a connection, in code |
| port | the number that picks which program on a host gets the connection |
| bind address | which interface(s) a server listens on: `127.0.0.1` or `0.0.0.0` |
| client / server | the side that connects / the side that listens |
| HTTP | a text request/response format on top of TCP |
| TLS | encryption and server identity wrapped around the TCP connection |
| REST API | HTTP endpoints addressed by URL, usually speaking JSON |
| SDK | a library that hides an API's HTTP behind function calls (`InventoryClient`) |
| idempotent | safe to repeat: doing it twice has the same effect as once |
| exponential backoff | waiting 1x, 2x, 4x... between retries |
| embedded database | a database that's a library in your process (SQLite) |
| client-server database | a database that's a separate server you connect to (Postgres) |
| container | a process isolated with its own filesystem and network |
| port mapping | NAT from a host port to a container port |
| volume | storage that outlives the container |
| columnar format | stores each column together (Parquet) |
| schema | column names and types, travelling with the data |
| latency | the time a round trip takes |
| connection pooling | keeping connections open for reuse (a `Session`) |

## Commit checkpoint

```bash
git status
git add pyproject.toml uv.lock README.md services/ src/netlab/ scripts/ notebooks/03_clients_servers_and_data.ipynb
git commit -m "notebook 03: echo server, inventory API endpoints, client, collector"
```

Check before you commit: `.env` must **not** appear in `git status` (it's gitignored; your token
lives there). Nothing from `data/` should appear either.

## What to build next

A small tool of your own that combines these pieces, chosen by you. Something you'd actually use:
a script that pulls something from an API you care about, keeps it in SQLite, and reports what
changed; or a service that answers a question about your network. Start with a README that says
what it does and the one command that runs it.